# Fine-to-Coarse Sequential Emotion Recognition — Kaggle pipeline

Source comes from the attached private dataset (`f2c-src`). Requires **GPU** + **Internet**
(GoEmotions from the Hugging Face Hub, MELD CSVs from GitHub).

Stages run in order and each writes to `/kaggle/working/{checkpoints,results}`, which Kaggle
keeps as the kernel output. Set `STAGES` below to run a subset.

In [ ]:
import os, shutil, subprocess, sys, time, pathlib

SRC = "/kaggle/input/f2c-src"
WORK = "/kaggle/working"
STAGES = ["stage1", "M1", "M2", "M3", "M4"]   # edit to run a subset
EXTRA = []                                     # e.g. ["--set", "epochs=3"]

for name in ("src", "scripts", "configs"):
    target = pathlib.Path(WORK, name)
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(pathlib.Path(SRC, name), target)
os.chdir(WORK)
sys.path.insert(0, f"{WORK}/src")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "transformers>=4.38" "datasets>=2.16" "scikit-learn>=1.4" "pyyaml>=6.0" 2>&1 | tail -2
import torch, transformers
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| transformers", transformers.__version__)

In [ ]:
def run(cmd):
    """Stream a pipeline step and fail loudly so the kernel stops on error."""
    print("\n>>>", " ".join(cmd), flush=True)
    started = time.time()
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        if "it/s]" in line or "s/it]" in line:      # collapse tqdm spam
            continue
        print(line, end="", flush=True)
    code_ = process.wait()
    print(f"<<< exit={code_} after {(time.time() - started) / 60:.1f} min", flush=True)
    if code_ != 0:
        raise RuntimeError(f"step failed: {' '.join(cmd)}")

## 1. Data

In [ ]:
run([sys.executable, "scripts/prepare_data.py", "--dataset", "all"])

## 2. Stage I — GoEmotions, 27 fine-grained classes

In [ ]:
if "stage1" in STAGES:
    run([sys.executable, "scripts/train_stage1.py", "--config", "configs/stage1.yaml", *EXTRA])

## 3. Stage II — systems M1-M4

In [ ]:
for system in [s for s in STAGES if s.startswith("M")]:
    run([sys.executable, "scripts/train_stage2.py", "--system", system, *EXTRA])

## 4. Tables and error analysis

In [ ]:
run([sys.executable, "scripts/build_results_table.py"])
run([sys.executable, "scripts/error_analysis.py"])

In [ ]:
import json

for path in sorted(pathlib.Path("results").glob("*/metrics.json")):
    payload = json.loads(path.read_text())
    if payload.get("stage") != "II":
        continue
    test = payload["test"]
    print(f"{payload['system']:>4}  acc={test['accuracy']:.4f}  "
          f"macro-F1={test['macro_f1']:.4f}  weighted-F1={test['weighted_f1']:.4f}")

## 5. Trim the output

Kaggle caps kernel output at 20 GB and uploads everything in `/kaggle/working`. Model weights are
large and re-trainable; metrics, predictions and tables are what the report needs.

In [ ]:
KEEP_CHECKPOINTS = False   # True keeps the Stage-I backbone for reuse (~500 MB)

for junk in ("src", "scripts", "configs", "data", "__pycache__"):
    shutil.rmtree(pathlib.Path(WORK, junk), ignore_errors=True)
if not KEEP_CHECKPOINTS:
    shutil.rmtree(pathlib.Path(WORK, "checkpoints"), ignore_errors=True)

for path in sorted(pathlib.Path(WORK).rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:8.2f} MB  {path.relative_to(WORK)}")